In [1]:
import os, pathlib
import pandas as pd

df = pd.read_csv("./CEAS_08.csv")
# df.head()



# Take a subset of the dataset (20% of each label)
df_label_0 = df[df['label'] == 0].sample(frac=0.001, random_state=42)
df_label_1 = df[df['label'] == 1].sample(frac=0.001, random_state=42)

df = pd.concat([df_label_0, df_label_1]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"New dataset size: {len(df)}")
print("Distribution of labels in the subset:")
print(df['label'].value_counts())


New dataset size: 39
Distribution of labels in the subset:
label
1    22
0    17
Name: count, dtype: int64


Imports & Setup

In [3]:
from __future__ import annotations
import os

# Quieter TensorFlow logs: 0=all, 1=INFO off, 2=WARNING off, 3=ERROR only
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import gc
import math
import random
from dataclasses import dataclass
from typing import List, Tuple, Optional

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset as TorchDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    set_seed,
)

# TextAttack imports
from textattack.models.wrappers import HuggingFaceModelWrapper
from textattack.attack_recipes import TextFoolerJin2019
from textattack import Attacker
from textattack.datasets import Dataset as TADataset


2025-08-28 13:48:56.250645: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1756367336.265983 1425607 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1756367336.271185 1425607 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1756367336.285437 1425607 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1756367336.285460 1425607 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1756367336.285462 1425607 computation_placer.cc:177] computation placer alr

GPU Sanity Check

In [4]:
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("Device name:", torch.cuda.get_device_name(0))

Torch version: 2.8.0+cu128
CUDA available: True
Device count: 1
Device name: NVIDIA GeForce RTX 4060 Ti


Dataset Utilities

In [5]:
@dataclass
class EncodedBatch:
    input_ids: torch.Tensor
    attention_mask: torch.Tensor
    labels: torch.Tensor


class TextClassificationDataset(TorchDataset):
    def __init__(self, texts: List[str], labels: List[int], tokenizer, max_length: int = 256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = int(self.labels[idx])
        enc = self.tokenizer(
            text,
            truncation=True,
            padding=False,
            max_length=self.max_length,
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(label, dtype=torch.long)
        return item


metrics

In [6]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted"),
    }


albert-based classifier

In [7]:
def train_discriminator(
    df: pd.DataFrame,
    save_dir: str = "./albert_phishing_model",
    model_name: str = "albert-base-v2",
    test_size: float = 0.2,
    max_length: int = 256,
    train_batch_size: int = 16,
    eval_batch_size: int = 32,
    learning_rate: float = 5e-5,
    num_train_epochs: int = 3,
    gradient_accumulation_steps: int = 1,
    fp16: bool = True,
    weight_decay: float = 0.01,
    logging_steps: int = 50,
    seed: int = 42,
    from_checkpoint: bool = False,
):
    """Train or resume a PyTorch ALBERT classifier with HuggingFace Trainer.

    Returns: (model, tokenizer, trainer)
    """
    assert {"body", "label"}.issubset(df.columns), "DataFrame must have 'body' and 'label' columns."

    os.makedirs(save_dir, exist_ok=True)
    set_seed(seed)

    # Split
    X_train, X_val, y_train, y_val = train_test_split(
        df["body"].astype(str).tolist(), df["label"].astype(int).tolist(), test_size=test_size, random_state=seed
    )

    # Load tokenizer & model
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if from_checkpoint and os.path.isdir(save_dir) and len(os.listdir(save_dir)) > 0:
        model = AutoModelForSequenceClassification.from_pretrained(save_dir)
    else:
        # Infer number of labels from data
        num_labels = int(pd.Series(y_train + y_val).nunique())
        model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

    # Datasets
    train_ds = TextClassificationDataset(X_train, y_train, tokenizer, max_length)
    val_ds = TextClassificationDataset(X_val, y_val, tokenizer, max_length)

    # Training args
    args = TrainingArguments(
        output_dir=save_dir,
        per_device_train_batch_size=train_batch_size,
        per_device_eval_batch_size=eval_batch_size,
        learning_rate=learning_rate,
        num_train_epochs=num_train_epochs,
        gradient_accumulation_steps=gradient_accumulation_steps,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        fp16=fp16,
        weight_decay=weight_decay,
        logging_steps=logging_steps,
        report_to=[],  # disable wandb etc.
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    # Save best model + tokenizer
    trainer.save_model(save_dir)
    tokenizer.save_pretrained(save_dir)

    # Cleanup
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return model, tokenizer, trainer

adversarial sample generation (TextAttack)

In [8]:
def generate_adversarial_samples(
    model: torch.nn.Module,
    tokenizer,
    dataset_df: pd.DataFrame,
    max_length: int = 256,
    recipe: str = "textfooler",
    seed: int = 42,
) -> pd.DataFrame:
    """Generate adversarial samples using TextAttack (untargeted misclassification).

    Returns a DataFrame with columns ['body','label'] using the original labels.
    """
    assert {"body", "label"}.issubset(dataset_df.columns)

    set_seed(seed)

    # Prepare TextAttack dataset (list of (text, label))
    tuples: List[Tuple[str, int]] = [
        (str(row["body"]), int(row["label"])) for _, row in dataset_df.iterrows()
    ]

    ta_dataset = TADataset(tuples)

    # Wrap model for TextAttack
    model.eval()
    if torch.cuda.is_available():
        model.to("cuda")
    wrapper = HuggingFaceModelWrapper(model, tokenizer)

    # Select recipe
    if recipe.lower() == "textfooler":
        attack = TextFoolerJin2019.build(wrapper)
    else:
        raise ValueError(f"Unsupported recipe: {recipe}")

    attacker = Attacker(attack, ta_dataset)

    adv_rows = []
    for result in attacker.attack_dataset():
        if getattr(result, "perturbed_result", None) is not None:
            adv_text = result.perturbed_result.attacked_text.text
            original_label = int(result.original_result.ground_truth_output)
            adv_rows.append({"body": adv_text, "label": original_label})

    adv_df = pd.DataFrame(adv_rows, columns=["body", "label"]).dropna()
    return adv_df


multi-round adversarial training game

In [9]:
def adversarial_training_game(
    df: pd.DataFrame,
    rounds: int = 5,
    save_dir: str = "./albert_game_model",
    model_name: str = "albert-base-v2",
    max_length: int = 256,
    train_batch_size: int = 16,
    eval_batch_size: int = 32,
    learning_rate: float = 5e-5,
    num_train_epochs: int = 3,
    gradient_accumulation_steps: int = 1,
    fp16: bool = True,
    weight_decay: float = 0.01,
    logging_steps: int = 50,
    seed: int = 42,
):
    """Run multi-round adversarial training using TextFooler.

    At each round:
      1) Train/update classifier
      2) Generate adversarial examples against the current model (on ALL data)
      3) Augment dataset and dedupe
      4) Save adversarial set for that round

    Returns: (final_model, final_tokenizer, final_df)
    """
    assert {"body", "label"}.issubset(df.columns)
    df = df.copy()
    os.makedirs(save_dir, exist_ok=True)

    for r in range(1, rounds + 1):
        print(f"\n===== Round {r} / {rounds} =====")

        # 1) Train or resume
        model, tokenizer, _ = train_discriminator(
            df=df,
            save_dir=save_dir,
            model_name=model_name,
            test_size=0.2,
            max_length=max_length,
            train_batch_size=train_batch_size,
            eval_batch_size=eval_batch_size,
            learning_rate=learning_rate,
            num_train_epochs=num_train_epochs,
            gradient_accumulation_steps=gradient_accumulation_steps,
            fp16=fp16,
            weight_decay=weight_decay,
            logging_steps=logging_steps,
            seed=seed,
            from_checkpoint=(r > 1),
        )

        # 2) Generate adversarials on the entire dataset
        adv_df = generate_adversarial_samples(
            model=model,
            tokenizer=tokenizer,
            dataset_df=df,
            max_length=max_length,
            recipe="textfooler",
            seed=seed,
        )
        print(f"Generated {len(adv_df)} adversarial samples.")

        # 3) Merge & dedupe
        if not adv_df.empty:
            before = len(df)
            df = pd.concat([df, adv_df], ignore_index=True)
            df.drop_duplicates(subset=["body"], inplace=True)
            after = len(df)
            print(f"Dataset size: {before} -> {after} (after merging adversarials)")
        else:
            print("No adversarial samples generated this round.")

        # 4) Save the adversarial set for this round
        adv_path = os.path.join(save_dir, f"adversarial_round_{r}.csv")
        adv_df.to_csv(adv_path, index=False)
        print(f"Saved adversarial samples to {adv_path}")

        # Cleanup GPU memory between rounds
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return model, tokenizer, df

In [10]:
import nltk
nltk.download("averaged_perceptron_tagger_eng")
nltk.download("wordnet")   # also needed for synonyms
nltk.download("omw-1.4")   # WordNet dependencies

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /home/nazmul/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to /home/nazmul/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/nazmul/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [ ]:
final_model, final_tokenizer, final_dataset = adversarial_training_game(
        df,
        rounds=3,
        save_dir="./albert_game_model",
        model_name="albert-base-v2",
        max_length=256,
        train_batch_size=8,
        eval_batch_size=32,
        learning_rate=5e-5,
        num_train_epochs=3,
        gradient_accumulation_steps=1,
        fp16=True,
        weight_decay=0.01,
        logging_steps=50,
        seed=42,
)


===== Round 1 / 3 =====


Some weights of AlbertForSequenceClassification were not initialized from the model checkpoint at albert-base-v2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_1425607/3098119841.py:64: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.652557,0.500000,0.333333
2,No log,0.587753,0.500000,0.333333
3,No log,0.471039,0.750000,0.733333


textattack: Unknown if model of class <class 'transformers.models.albert.modeling_albert.AlbertForSequenceClassification'> compatible with goal function <class 'textattack.goal_functions.classification.untargeted_classification.UntargetedClassification'>.


Attack(
  (search_method): GreedyWordSwapWIR(
    (wir_method):  delete
  )
  (goal_function):  UntargetedClassification
  (transformation):  WordSwapEmbedding(
    (max_candidates):  50
    (embedding):  WordEmbedding
  )
  (constraints): 
    (0): WordEmbeddingDistance(
        (embedding):  WordEmbedding
        (min_cos_sim):  0.5
        (cased):  False
        (include_unknown_words):  True
        (compare_against_original):  True
      )
    (1): PartOfSpeech(
        (tagger_type):  nltk
        (tagset):  universal
        (allow_verb_noun_swap):  True
        (compare_against_original):  True
      )
    (2): UniversalSentenceEncoder(
        (metric):  angular
        (threshold):  0.840845057
        (window_size):  15
        (skip_text_shorter_than_window):  True
        (compare_against_original):  False
      )
    (3): RepeatModification
    (4): StopwordModification
    (5): InputColumnModification(
        (matching_column_labels):  ['premise', 'hypothesis']
       

...

--------------------------------------------- Result 9 ---------------------------------------------
[[1 (65%)]] --> [[0 (50%)]]

[[This]] message is intended for :

*[[SAVE]]!  [[SAVE]]!  [[SAVE]]!*

[[LOW]] [[COST]] GICENRES
- [[No]] Doctor Fees Or [[Prescriptions]] Needed
- [[Lower]] your [[monthly]] medication [[expensive]]
- Many [[shipping]] [[options]] available

http://fameideal.com


Grove's daily talks, as [[well]] as other perquisites. [[Members]] may also invite guests to the [[Grove]] although

[[Use]] http://fameideal.com/a.php for [[removal]]


[[That]] message is intended for :

*[[ECONOMIES]]!  [[ASCETICISM]]!  [[CONSERVATION]]!*

[[MARGINALLY]] [[COSTING]] GICENRES
- [[Absence]] Doctor Fees Or [[Statutes]] Needed
- [[Diminish]] your [[weekly]] medication [[costly]]
- Many [[maritime]] [[possibility]] available

http://fameideal.com


Grove's daily talks, as [[satisfactorily]] as other perquisites. [[Delegated]] may also invite guests to the [[Orchards]] although

[[Ut

[Succeeded / Failed / Skipped / Total] 5 / 4 / 1 / 10: 100%|██████████| 10/10 [03:02<00:00, 18.21s/it]

--------------------------------------------- Result 10 ---------------------------------------------
[[0 (80%)]] --> [[[FAILED]]]

>Thus I'm concluding if it doesn't work then it might be a BIOS issue. Once again:  - The Bios has to reserve address space for PCI devices and usually     reserves between 0.5 and 1 GiB.  - To make the physical RAM that's hidden by that reservation     available, most BIOSs map this RAM beyond the 4 GiB (2^32)    Threshold. - To address that memory, you need a kernel supporting either    PAE or long (i.e. 64 bit) mode. If the memory isn't available, it's rather obvious that it is the BIOS that failed to do the mapping. >In PAE the operating system uses page tables to map this 4 GiB >address space onto the 64 GiB of total memory, and the map is >usually different for each process. You've got something mixed up there. Not the OS but the processor uses page tables and it does that for *all* operating modes, not only PAE, as virtual memory management on x86 a


Generated 10 adversarial samples.
Dataset size: 58 -> 67 (after merging adversarials)
Saved adversarial samples to ./albert_game_model/adversarial_round_3.csv


: 